# Week 2 Exercise — Technical Tutor (Anthropic edition)

The end-of-week challenge: a **technical question-answerer** that brings together everything from Week 2 — a Gradio UI, streaming, a system prompt that adds expertise, and **model switching** — plus the two bonuses: **tool use** and **audio in/out**.

This is a working reference you can run and extend. Targets **Gradio 6**. Sections:
1. Core: streaming technical tutor with Claude / Ollama switching
2. Bonus 1: a tool (look up the latest version of a PyPI package)
3. Bonus 2 (optional): talk to it (Whisper) and hear it reply (gTTS)

Keep this notebook's deps installed: `pip install anthropic openai python-dotenv gradio gtts`

## Setup

In [11]:
import os
import requests
from dotenv import load_dotenv
import anthropic
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
print("Anthropic key set" if os.getenv("ANTHROPIC_API_KEY") else "Anthropic key NOT set")

client = anthropic.Anthropic()                                          # Claude
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1") # free local

Anthropic key set


In [12]:
def to_text(content):
    """Flatten Gradio-6 block content into a plain string (safe on older Gradio too)."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    return str(content)

## The expertise lives in the system prompt

This is the lever that turns a generic chatbot into a *technical tutor*. Tweak the persona to taste.

In [4]:
EXPERT_SYSTEM = """
You are a patient, expert software engineering tutor.
When given a technical question or a snippet of code, explain it clearly:
- start with a one-sentence plain-English summary,
- then break down the key parts step by step,
- show a short corrected or illustrative code example when useful,
- and end with one common pitfall to watch for.
Use markdown. Be concise but complete. If the question is ambiguous, state your assumption and answer anyway.
"""

## Core: streaming tutor with model switching

`additional_inputs` lets a `ChatInterface` carry an extra control (the model dropdown) alongside the chat box. The function streams from whichever model is selected — Claude tiers or local Ollama.

In [5]:
def stream_answer(message, history, model_choice):
    messages = [{"role": h["role"], "content": to_text(h["content"])} for h in history]
    messages.append({"role": "user", "content": message})

    if model_choice.startswith("Claude"):
        model = "claude-sonnet-4-6" if "Sonnet" in model_choice else "claude-haiku-4-5"
        with client.messages.stream(model=model, max_tokens=1500, system=EXPERT_SYSTEM, messages=messages) as stream:
            partial = ""
            for text in stream.text_stream:
                partial += text
                yield partial
    else:  # Ollama (free, local) - OpenAI-style, system goes in the list here
        omsgs = [{"role": "system", "content": EXPERT_SYSTEM}] + messages
        stream = ollama.chat.completions.create(model="llama3.2", messages=omsgs, stream=True)
        partial = ""
        for chunk in stream:
            partial += chunk.choices[0].delta.content or ""
            yield partial

In [ ]:
model_selector = gr.Dropdown(
    ["Claude (Sonnet)", "Claude (Haiku)", "Ollama (llama3.2)"],
    value="Claude (Sonnet)", label="Model",
)

gr.ChatInterface(
    fn=stream_answer,
    additional_inputs=[model_selector],
    title="Technical Tutor",
    description="Ask a coding question or paste a snippet. Switch models with the dropdown.",
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 

## Bonus 1: give the tutor a tool

A tutor often needs *current* facts — like the latest version of a library, which the model can't know reliably. We add a tool that queries PyPI's public JSON API (no key needed). When the user asks "what's the latest version of pandas?", Claude calls the tool instead of guessing.

In [7]:
def get_latest_package_version(package_name):
    print(f"TOOL CALLED: looking up {package_name} on PyPI", flush=True)
    try:
        r = requests.get(f"https://pypi.org/pypi/{package_name}/json", timeout=10)
        if r.status_code == 200:
            return f"The latest version of {package_name} is {r.json()['info']['version']}"
        return f"No package named '{package_name}' was found on PyPI"
    except Exception as e:
        return f"Error looking up {package_name}: {e}"

get_latest_package_version("pandas")

TOOL CALLED: looking up pandas on PyPI


'The latest version of pandas is 3.0.3'

In [8]:
version_tool = {
    "name": "get_latest_package_version",
    "description": "Get the latest released version of a Python package from PyPI. "
                   "Use whenever the user asks about the current or latest version of a package.",
    "input_schema": {
        "type": "object",
        "properties": {
            "package_name": {"type": "string", "description": "The PyPI package name, e.g. 'pandas'"},
        },
        "required": ["package_name"],
    },
}
tools = [version_tool]

The tool path is non-streaming (you can't cleanly stream while doing tool round-trips). Same Anthropic loop you mastered in Day 4: ask, run, return results as a **user** turn, repeat.

In [9]:
def chat_with_tools(message, history):
    messages = [{"role": h["role"], "content": to_text(h["content"])} for h in history]
    messages.append({"role": "user", "content": message})

    response = client.messages.create(model="claude-sonnet-4-6", max_tokens=1500,
                                      system=EXPERT_SYSTEM, messages=messages, tools=tools)
    while response.stop_reason == "tool_use":
        results = []
        for block in response.content:
            if block.type == "tool_use" and block.name == "get_latest_package_version":
                out = get_latest_package_version(block.input["package_name"])   # input is a dict
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": out})
        messages.append({"role": "assistant", "content": response.content})     # echo request
        messages.append({"role": "user", "content": results})                   # results as USER turn
        response = client.messages.create(model="claude-sonnet-4-6", max_tokens=1500,
                                          system=EXPERT_SYSTEM, messages=messages, tools=tools)

    return "".join(b.text for b in response.content if b.type == "text")

In [10]:
gr.ChatInterface(
    fn=chat_with_tools,
    title="Technical Tutor (with PyPI tool)",
    description="Try: 'What's the latest version of fastapi, and what's new conceptually in modern versions?'",
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


TOOL CALLED: looking up python on PyPI


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


TOOL CALLED: looking up fastapi on PyPI


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


TOOL CALLED: looking up fastapi on PyPI


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Bonus 2 (optional): talk to it, and hear it reply

Claude is text-only, so we borrow free specialist models for the audio ends:
- **Hear it** → gTTS (free, no key).
- **Talk to it** → OpenAI's **Whisper**, run locally (`pip install openai-whisper`). It's a big-ish download; this section is optional.

This is the production pattern: Claude reasons in the middle; specialist models handle speech-to-text and text-to-speech on either side.

In [ ]:
# Audio OUT - gTTS
from gtts import gTTS
import tempfile

def speak(text):
    path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    gTTS(text=text, lang="en").save(path)
    return path

In [ ]:
# Audio IN - Whisper (optional, heavy). Uncomment to enable.
# import whisper
# stt_model = whisper.load_model("base")   # downloads once
# def transcribe(audio_path):
#     return stt_model.transcribe(audio_path)["text"]

A Blocks UI wiring mic → transcribe → tutor → speech. (Requires the Whisper cell above to be enabled. If you skip Whisper, type into the textbox instead.)

In [ ]:
def voice_turn(audio_path, history):
    # question = transcribe(audio_path)        # <- enable with Whisper
    question = "(enable Whisper to transcribe speech)"
    history = list(history) + [{"role": "user", "content": question}]
    answer = chat_with_tools(question, history[:-1])
    history.append({"role": "assistant", "content": answer})
    return history, speak(answer)

with gr.Blocks() as voice_ui:
    gr.Markdown("## Talk to your Technical Tutor")
    chatbot = gr.Chatbot(type="messages", height=400)
    mic = gr.Audio(sources="microphone", type="filepath", label="Ask a question")
    reply_audio = gr.Audio(autoplay=True, label="Tutor's reply")
    mic.stop_recording(voice_turn, inputs=[mic, chatbot], outputs=[chatbot, reply_audio])

voice_ui.launch(inbrowser=True)

## Extension ideas

- Add a **second tool** (e.g. fetch a function's signature, or run a sandboxed snippet) — the loop already supports any number.
- Persist conversations to SQLite so the tutor remembers across sessions.
- Add a **difficulty toggle** (beginner / advanced) as another `additional_inputs` control that swaps the system prompt.
- This is a real, demonstrable portfolio piece: a multi-model, tool-using, voice-capable technical assistant. Commit it.